In [8]:
import os
new_dir = "/home/jingqi/RNALocateV3.0"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi/RNALocateV3.0'

In [3]:
import scipy
import numpy as np
import scanpy as sc
import pandas as pd

In [11]:
file_path_1 = "Data/FIMO/a_aggregated_delicate.csv"
df_1 = pd.read_csv(file_path_1)
file_path_2 = "Data/Validation/transcript_localization.csv"
df_2 = pd.read_csv(file_path_2)

In [39]:
file_path_fimo = "Data/FIMO/result_all_transcripts/fimo.txt"
fimo_result = pd.read_csv(file_path_fimo, sep='\t')

In [40]:
print(fimo_result.columns.tolist())

['#pattern name', 'sequence name', 'start', 'stop', 'strand', 'score', 'p-value', 'q-value', 'matched sequence']


In [41]:
# Rename columns to match other dataframes
fimo_result = fimo_result.rename(columns={
    '#pattern name': 'Motif',
    'sequence name': 'Transcript'
})

print(fimo_result.columns.tolist())

['Motif', 'Transcript', 'start', 'stop', 'strand', 'score', 'p-value', 'q-value', 'matched sequence']


## Map the validated motifs onto transcripts of interest

In [54]:
# Filter FIMO results to ONLY include motifs of interest
# 'how=inner' ensures we only keep hits for the motifs present in df_1
merged_motifs = pd.merge(fimo_result, df_1, on='Motif', how='inner')

# Filter the remaining hits to ONLY include your transcripts of interest
# 'how=inner' ensures we only keep hits for the transcripts present in df_2
final_result = pd.merge(merged_motifs, df_2, on='Transcript', how='inner')

# Arrange columns cleanly (Keeping p-value and q-value as requested)
# We define a preferred order, but safely only include columns that actually exist
preferred_columns = [
    'Gene', 'Transcript', 'Motif', 'Cluster', 'strand', 'score', 'q-value',
    'Localization_1', 'Localization_2', 'Localization_3', 'Localization_4', 'Localization_5'
]

# Keep any extra columns FIMO might have generated just in case, pushed to the end
other_cols = [c for c in final_result.columns if c not in preferred_columns]
final_cols = [c for c in preferred_columns if c in final_result.columns]

final_result = final_result[final_cols]

output_path = "Data/Bridge_critical/raw_mapping.csv"
final_result.to_csv(output_path, index=False)
print("FIMO Mapping Complete")

FIMO Mapping Complete


In [53]:
# Filter by strand ('+') and drop the 'strand' column
df_refined = final_result[final_result['strand'] == '+'].copy()
# Filter by q-value (< 0.05) and drop the 'q-value' column
df_refined = df_refined[df_refined['q-value'] < 0.05]

# Combine all Localization columns into one
loc_cols = [c for c in df_refined.columns if 'Localization' in c]

# Join the non-null values row by row using a comma and space
df_refined['Localization'] = df_refined[loc_cols].apply(
    lambda row: ', '.join(row.dropna().astype(str)), axis=1
)

# Grouping and Aggregation 
group_cols = ['Gene', 'Transcript', 'Motif', 'Cluster', 'Localization']

# Define the custom function: 1 - product(1 - q)
def calc_significance(q_series):
    return 1 - np.prod(1 - q_series)

# Aggregate: This automatically drops those not explicitly included
df_agg = df_refined.groupby(group_cols).agg(
    Hits=('q-value', 'size'),
    Significance=('q-value', calc_significance)
).reset_index()

# Save the aggregated result
output_path = "Data/Bridge_critical/motif_transcript_matches.csv"
df_agg.to_csv(output_path, index=False)

# Report results
print(f"Aggregated rows (unique motif per transcript): {len(df_agg)}")

Aggregated rows (unique motif per transcript): 277
